# Ablation Study: Component Analysis of TE-Q-Transformer
## Controlled Empirical Experiments

**Research Objective:** Systematically evaluate and decouple the performance contributions of:
1. **Physics-Guided Arrhenius Temperature Encoding vs. Conventional Temperature Scaling** (E02)
2. **Degradation Kinetics Mechanisms: SEI-only vs. Plating-only vs. Combined** (E03)
3. **Variational Quantum Feature Map vs. Parameter-Matched Classical Counterpart** (E04)
4. **Trainable vs. Fixed Arrhenius Activation Energies & Architecture Modules**

---

### Key Manuscript Takeaways:
- Embedding degradation physics reduces macro RMSE from **0.0201 to 0.0168 (16.5% improvement)** compared to raw temperature.
- Replacing the 4-qubit quantum circuit with a classical MLP counterpart of equal parameter capacity raises macro RMSE from **0.0215 to 0.0431**.


In [ ]:
# ==============================================================================
# 0. CONFIGURABLE REPOSITORY ROOT PATH & ENVIRONMENT SETUP
# ==============================================================================
import os
import sys
from pathlib import Path

# Manual override: Set to Path("your/path") if needed; otherwise auto-discovered.
MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / REPO_NAME,
    Path("/kaggle/working"),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
DATA_ROOT = REPO_ROOT / "datasets"
MODEL_ROOT = REPO_ROOT / "models"
RESULT_ROOT = REPO_ROOT / "results"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Install PennyLane if running in a fresh cloud runtime (Kaggle/Colab)
try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")


## 1. Ablation E02: Conventional vs. Physics-Guided Temperature Encoding
Under identical 3-layer Transformer and 4-qubit quantum circuit settings:
- **Condition A (Raw Temperature):** Temperature is linearly normalized alongside voltage, current, and time into $[0, 1]$.
- **Condition B (Physics-Guided):** Temperature remains in physical Celsius to drive the dual-mechanism Arrhenius transformation $\phi(T) = \exp\left(\frac{E_{a,\text{sei}}}{R}\left(\frac{1}{T_{\text{ref}}} - \frac{1}{T}\right)\right) + \exp\left(\frac{E_{a,\text{pl}}}{R}\left(\frac{1}{T} - \frac{1}{T_{\text{ref}}}\right)\right)$.


In [ ]:
e02_csv = RESULT_ROOT / "tables" / "ablation_raw_vs_physics.csv"
if e02_csv.exists():
    df_e02 = pd.read_csv(e02_csv)
    print("[Ablation E02 Results: Raw Temperature vs Physics-Guided Arrhenius]")
    display(df_e02)
    
    # Plot RMSE comparison
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(df_e02['cell']))
    width = 0.35
    ax.bar(x - width/2, df_e02['Raw_RMSE'], width, label="Raw Temperature Scaling", color="#E07A5F")
    ax.bar(x + width/2, df_e02['Physics_RMSE'], width, label="Physics-Guided Arrhenius", color="#2A9D8F")
    ax.set_xticks(x)
    ax.set_xticklabels(df_e02['cell'])
    ax.set_ylabel("RMSE")
    ax.set_title("Ablation E02: Effect of Physics-Guided Temperature Encoding")
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print(f"Table not found at {e02_csv}")


## 2. Ablation E03: Degradation Mechanism Analysis
Decomposing the Arrhenius thermal formulation:
1. **SEI Growth Only ($E_{a,\text{sei}}$):** Dominant at elevated temperatures ($T > T_{\text{ref}}$).
2. **Lithium Plating Only ($E_{a,\text{pl}}$):** Dominant at sub-ambient temperatures ($T < T_{\text{ref}}$).
3. **Combined Dual Mechanism:** Both mechanisms active simultaneously.


In [ ]:
e03_csv = RESULT_ROOT / "tables" / "ablation_mechanisms.csv"
if e03_csv.exists():
    df_e03 = pd.read_csv(e03_csv)
    print("[Ablation E03 Results: Thermal Degradation Mechanisms]")
    display(df_e03)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(df_e03['cell']))
    w = 0.25
    ax.bar(x - w, df_e03['SEI_RMSE'], w, label="SEI-Only", color="#457B9D")
    ax.bar(x, df_e03['Plating_RMSE'], w, label="Plating-Only", color="#E76F51")
    ax.bar(x + w, df_e03['Combined_RMSE'], w, label="Combined (Dual-Mechanism)", color="#2A9D8F")
    ax.set_xticks(x)
    ax.set_xticklabels(df_e03['cell'])
    ax.set_ylabel("RMSE")
    ax.set_title("Ablation E03: Individual vs Combined Degradation Mechanisms")
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()


## 3. Ablation E04: Quantum vs. Parameter-Matched Classical Feature Map
Investigating whether the quantum circuit provides genuine representational value or merely acts as an arbitrary non-linear projection:
- **Quantum:** 4-qubit simulated circuit with rich entangler (RZ, CNOT, IsingZZ, Toffoli, Hadamard, RZ) and Pauli-Z expectation values.
- **Classical Counterpart:** Parameter-matched 2-layer MLP with GELU activations matching the 4-channel input and projection dimensions under identical physics encoding and Transformer encoder.


In [ ]:
e04_csv = RESULT_ROOT / "tables" / "ablation_classical_vs_quantum.csv"
if e04_csv.exists():
    df_e04 = pd.read_csv(e04_csv)
    print("[Ablation E04 Results: Classical MLP vs 4-Qubit Quantum Circuit]")
    display(df_e04)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(df_e04['cell']))
    w = 0.35
    ax.bar(x - w/2, df_e04['Classical_RMSE'], w, label="Classical Parameter-Matched MLP", color="#D62828")
    ax.bar(x + w/2, df_e04['Quantum_RMSE'], w, label="4-Qubit Quantum Feature Map", color="#003049")
    ax.set_xticks(x)
    ax.set_xticklabels(df_e04['cell'])
    ax.set_ylabel("RMSE")
    ax.set_title("Ablation E04: Classical MLP vs Quantum Feature Representation")
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
